# Cluster Image Tiling

This notebook creates a tiled grid of cluster cutouts with random color assignments:
- VIS filter → Blue channel
- NIR_H filter → Red channel  
- NIR_Y filter → Green channel

Each cluster is randomly assigned one of these three filters/colors to create a visually striking mosaic.

In [ ]:
# Import necessary packages
from astroquery.esa.euclid.core import EuclidClass
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
import matplotlib.pyplot as plt
from astropy.visualization import ImageNormalize, PercentileInterval, AsinhStretch
from astropy.io import fits
import numpy as np
from astropy.table import Table
from astropy.wcs import WCS
import os
from pathlib import Path
import random

In [ ]:
# Set up Euclid archive environment
Euclid = EuclidClass(environment='REG')

In [ ]:
# Login to Euclid archive
Euclid.login(credentials_file='/media/user/cred.txt')

## Load the Cluster Catalogue

In [ ]:
# Load cluster catalogue
clusters = Table(fits.open('EUC_LE3_DET-CL_CL-DETECTIONS-PZWAV_20250526T104242.195573Z_03.20.fits')[1].data)
print(f"Total clusters available: {len(clusters)}")

## Configuration

In [ ]:
# Configuration parameters
n_clusters = 100  # Number of clusters to use
fov = 1  # Field of view in arcminutes
grid_size = 10  # 10x10 grid = 100 tiles
tile_size = 600  # Target tile size in pixels (will crop/resize to this)
output_dir = Path('cutouts_tiling')
output_dir.mkdir(exist_ok=True)

# Filter to color mapping
filter_colors = {
    'VIS': ('blue', [0, 0, 1]),      # Blue channel
    'NIR_H': ('red', [1, 0, 0]),     # Red channel
    'NIR_Y': ('green', [0, 1, 0])    # Green channel
}

## Randomly Select Clusters and Assign Filters

In [ ]:
# Randomly select clusters
np.random.seed(42)  # For reproducibility
selected_indices = np.random.choice(len(clusters), size=n_clusters, replace=False)
selected_clusters = clusters[selected_indices]

# Randomly assign a filter to each cluster
filters = list(filter_colors.keys())
assigned_filters = [random.choice(filters) for _ in range(n_clusters)]

print(f"Selected {n_clusters} clusters")
print(f"Filter distribution: VIS={assigned_filters.count('VIS')}, "
      f"NIR_H={assigned_filters.count('NIR_H')}, NIR_Y={assigned_filters.count('NIR_Y')}")

## Download Cutouts

In [ ]:
def get_cluster_cutout(cluster, filter_name, cluster_idx, fov=3):
    """
    Download a cutout for a single cluster in the specified filter.
    
    Parameters:
    -----------
    cluster : astropy table row
        Cluster data
    filter_name : str
        Filter name (VIS, NIR_H, or NIR_Y)
    cluster_idx : int
        Index for naming the output file
    fov : float
        Field of view in arcminutes
    
    Returns:
    --------
    str or None
        Path to downloaded cutout, or None if failed
    """
    ra = cluster['RIGHT_ASCENSION_CLUSTER']
    dec = cluster['DECLINATION_CLUSTER']
    cluster_id = cluster['ID_CLUSTER']
    
    coord = SkyCoord(ra, dec, frame='icrs', unit='deg')
    radius = u.Quantity(0.5, u.deg)
    
    try:
        # Search for available images
        j = Euclid.cone_search(
            coordinate=coord,
            radius=radius,
            table_name="sedm.mosaic_product",
            ra_column_name="ra",
            dec_column_name="dec",
            columns="*",
            async_job=True
        )
        cone_results = j.get_results()
        
        # Filter for REGREPROC2 and the desired filter
        mask = (
            (cone_results['filter_name'] == filter_name) &
            (np.char.find(cone_results['file_path'].astype(str), 'REGREPROC2') >= 0)
        )
        
        if len(cone_results[mask]) > 0:
            example_file = cone_results[mask][0]
            
            output_path = output_dir / f'cluster_{cluster_idx:03d}_{filter_name}.fits'
            
            # Download cutout
            saved_cutout_filepath = Euclid.get_cutout(
                file_path=example_file["file_path"] + "/" + example_file["file_name"],
                instrument=example_file["instrument_name"],
                id=example_file["tile_index"],
                coordinate=coord,
                radius=fov * u.arcmin,
                output_file=str(output_path)
            )
            
            return str(output_path)
        else:
            print(f"No {filter_name} data found for cluster {cluster_id} at idx {cluster_idx}")
            return None
            
    except Exception as e:
        print(f"Error downloading cluster {cluster_id} at idx {cluster_idx}: {e}")
        return None

In [ ]:
# Download all cutouts
cutout_paths = []

for idx, (cluster, filter_name) in enumerate(zip(selected_clusters, assigned_filters)):
    print(f"Downloading {idx+1}/{n_clusters}: Cluster {cluster['ID_CLUSTER']} in {filter_name}...")
    path = get_cluster_cutout(cluster, filter_name, idx, fov=fov)
    cutout_paths.append((path, filter_name))

# Filter out failed downloads
successful_cutouts = [(p, f) for p, f in cutout_paths if p is not None]
print(f"\nSuccessfully downloaded {len(successful_cutouts)}/{n_clusters} cutouts")

## Create Tiled Grid

In [ ]:
def crop_or_pad_to_size(data, target_size):
    """
    Crop or pad an image to a target square size, centered.

    Parameters:
    -----------
    data : numpy array
        2D image array
    target_size : int
        Target size for both dimensions

    Returns:
    --------
    numpy array
        Cropped/padded image of size (target_size, target_size)
    """
    height, width = data.shape

    # Create output array
    output = np.zeros((target_size, target_size))

    # Calculate crop/pad for height
    if height > target_size:
        # Crop
        start_h = (height - target_size) // 2
        cropped_h = data[start_h:start_h + target_size, :]
    elif height < target_size:
        # Pad
        start_h = (target_size - height) // 2
        output[start_h:start_h + height, :] = data
        return output[:, :width] if width <= target_size else output
    else:
        cropped_h = data

    # Calculate crop/pad for width
    if width > target_size:
        # Crop
        start_w = (width - target_size) // 2
        output = cropped_h[:, start_w:start_w + target_size]
    elif width < target_size:
        # Pad
        start_w = (target_size - width) // 2
        output[:cropped_h.shape[0], start_w:start_w + width] = cropped_h
    else:
        output = cropped_h

    return output


def process_single_filter_cutout(fits_path, filter_name, target_size=600):
    """
    Process a single-filter cutout and convert to RGB with the appropriate color.

    Parameters:
    -----------
    fits_path : str
        Path to FITS file
    filter_name : str
        Filter name (VIS, NIR_H, NIR_Y)
    target_size : int
        Target size for the output tile (will crop/pad to this size)

    Returns:
    --------
    numpy array
        RGB image array of shape (target_size, target_size, 3)
    """
    # Load the data
    data = fits.getdata(fits_path)

    # Crop or pad to consistent size
    data_resized = crop_or_pad_to_size(data, target_size)

    # Scale the data
    transform = AsinhStretch() + PercentileInterval(99.5)
    scaled = transform(data_resized)

    # Get the RGB color for this filter
    color_name, rgb = filter_colors[filter_name]

    # Create RGB image by multiplying scaled data with color
    rgb_image = np.zeros((*scaled.shape, 3))
    for i in range(3):
        rgb_image[:, :, i] = scaled * rgb[i]

    return rgb_image

In [ ]:
# All tiles will be resized to a consistent size
if successful_cutouts:
    tile_height = tile_size
    tile_width = tile_size
    print(f"Target tile size: {tile_width} x {tile_height} pixels")
else:
    raise ValueError("No successful cutouts to create grid")

In [ ]:
# Calculate grid dimensions
n_successful = len(successful_cutouts)
grid_cols = grid_size
grid_rows = int(np.ceil(n_successful / grid_cols))

print(f"Creating {grid_rows} x {grid_cols} grid for {n_successful} clusters")

# Create empty grid
grid_image = np.zeros((grid_rows * tile_height, grid_cols * tile_width, 3))

# Fill the grid
for idx, (cutout_path, filter_name) in enumerate(successful_cutouts):
    row = idx // grid_cols
    col = idx % grid_cols
    
    print(f"Processing tile {idx+1}/{n_successful} (row={row}, col={col}, filter={filter_name})")
    
    # Process the cutout with target size
    rgb_tile = process_single_filter_cutout(cutout_path, filter_name, target_size=tile_size)
    
    # Place in grid
    y_start = row * tile_height
    y_end = y_start + tile_height
    x_start = col * tile_width
    x_end = x_start + tile_width
    
    grid_image[y_start:y_end, x_start:x_end, :] = rgb_tile

print("Grid creation complete!")

## Display and Save the Tiled Image

In [ ]:
# Display the grid
fig, ax = plt.subplots(figsize=(20, 20))
ax.imshow(grid_image, origin='lower')
ax.set_title(f'Euclid Cluster Mosaic: {n_successful} Clusters', fontsize=20)
ax.axis('off')
plt.tight_layout()
plt.savefig('cluster_mosaic_grid.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved mosaic to cluster_mosaic_grid.png")

In [ ]:
# Optional: Create a version with grid lines
fig, ax = plt.subplots(figsize=(20, 20))
ax.imshow(grid_image, origin='lower')
ax.set_title(f'Euclid Cluster Mosaic: {n_successful} Clusters (with grid)', fontsize=20)

# Add grid lines
for i in range(1, grid_rows):
    ax.axhline(y=i * tile_height, color='white', linewidth=0.5, alpha=0.3)
for j in range(1, grid_cols):
    ax.axvline(x=j * tile_width, color='white', linewidth=0.5, alpha=0.3)

ax.axis('off')
plt.tight_layout()
plt.savefig('cluster_mosaic_grid_with_lines.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved mosaic with grid lines to cluster_mosaic_grid_with_lines.png")

## Summary Statistics

In [ ]:
# Count filters in successful cutouts
filter_counts = {f: 0 for f in filter_colors.keys()}
for _, filter_name in successful_cutouts:
    filter_counts[filter_name] += 1

print("\n=== Mosaic Summary ===")
print(f"Total tiles: {n_successful}")
print(f"Grid size: {grid_rows} x {grid_cols}")
print(f"Final image size: {grid_image.shape[1]} x {grid_image.shape[0]} pixels")
print(f"\nFilter distribution:")
for filter_name, count in filter_counts.items():
    color_name, _ = filter_colors[filter_name]
    print(f"  {filter_name} ({color_name}): {count} tiles ({100*count/n_successful:.1f}%)")